# Lab: Tool-Using AI Agent

**Scenario:** Amr Store Assistant - an e-commerce support agent

Requirements implemented:
- Has own resources (2 local files: shipping & returns policy)
- Has online search tool (Wikipedia API)
- Has tool to query local database (SQLite products table)
- Strong system prompt to use tools only when needed

### 1. Install Dependencies

In [ ]:
!pip install langchain langchain-openai langgraph python-dotenv requests

### 2. Setup Local Resources (2 Files)

In [ ]:
from pathlib import Path

RESOURCE_DIR = Path("resources")
RESOURCE_DIR.mkdir(exist_ok=True)

shipping_policy = """Shipping Policy
================
1) Standard shipping takes 3 to 5 business days.
2) Express shipping takes 1 to 2 business days.
3) Free standard shipping is available for orders above $75.
4) International shipping currently supports Egypt, Saudi Arabia, and UAE.
5) Orders are processed Monday to Friday, excluding public holidays.
"""

returns_policy = """Returns Policy
==============
1) Products can be returned within 14 days from delivery date.
2) Items must be unused and in original packaging.
3) Refunds are processed within 5 to 7 business days after inspection.
4) Shipping fees are non-refundable unless the item is defective.
5) Damaged or wrong products must be reported within 48 hours of delivery.
"""

(RESOURCE_DIR / "shipping_policy.txt").write_text(shipping_policy)
(RESOURCE_DIR / "returns_policy.txt").write_text(returns_policy)

print("Created resource files:")
for f in RESOURCE_DIR.glob("*.txt"):
    print(f"  - {f}")

### 3. Setup Local SQLite Database

In [ ]:
import sqlite3

DB_PATH = Path("store.db")

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        stock INTEGER NOT NULL
    )
""")

cur.execute("DELETE FROM products")
cur.executemany(
    "INSERT INTO products (name, category, price, stock) VALUES (?, ?, ?, ?)",
    [
        ("Wireless Mouse", "Accessories", 19.99, 42),
        ("Mechanical Keyboard", "Accessories", 69.50, 15),
        ("USB-C Hub", "Accessories", 34.99, 26),
        ("27-inch Monitor", "Displays", 229.00, 8),
        ("Laptop Stand", "Office", 24.90, 31),
    ],
)

conn.commit()
conn.close()
print(f"Database initialized at: {DB_PATH}")

### 4. Define Tools

In [ ]:
import sqlite3
import requests
from pathlib import Path
from langchain.tools import tool

RESOURCE_DIR = Path("resources")
DB_PATH = Path("store.db")


@tool
def read_local_resources(topic: str) -> str:
    """
    Search inside local policy files (shipping_policy.txt, returns_policy.txt).
    MUST use this tool for ANY question about: shipping, delivery, returns, refunds, policy, international orders.
    Input: keyword like 'shipping', 'return', 'refund', 'international', 'express'.
    """
    topic_lower = topic.lower().strip()
    files = list(RESOURCE_DIR.glob("*.txt"))

    matches = []
    for file_path in files:
        lines = file_path.read_text(encoding="utf-8").splitlines()
        for line in lines:
            if topic_lower in line.lower():
                matches.append(f"{file_path.name}: {line}")

    if not matches:
        return "No direct match found. Try keywords like: shipping, return, refund, international."
    return "\n".join(matches[:15])


@tool
def search_online(query: str) -> str:
    """
    Search online using Wikipedia API.
    Use this only when the question requires external/general web knowledge.
    """
    q = query.strip()
    if not q:
        return "Empty query."

    params = {
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json",
        "utf8": 1,
    }

    try:
        response = requests.get("https://en.wikipedia.org/w/api.php", params=params, timeout=10)
        response.raise_for_status()
        results = response.json().get("query", {}).get("search", [])
    except Exception as exc:
        return f"Online search failed: {exc}"

    if not results:
        return "No online result found."

    top = results[0]
    title = top.get("title", "Unknown")
    snippet = top.get("snippet", "").replace('<span class="searchmatch">', "").replace("</span>", "")
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"

    return f"Title: {title}\nSnippet: {snippet}\nURL: {url}"


@tool
def search_local_database(query: str) -> str:
    """
    Search products table in local SQLite database.
    MUST use this tool for ANY question about: products, stock, inventory, price, availability, items, accessories, monitors, keyboards, mouse, or catalog.
    Input: product name, category, or keyword. Output: matching products with price and stock.
    """
    if not DB_PATH.exists():
        return "Database not found. Run setup cell first."

    q = query.strip().lower()
    if not q:
        return "Empty query."

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        """
        SELECT name, category, price, stock
        FROM products
        WHERE lower(name) LIKE ? OR lower(category) LIKE ?
        ORDER BY stock DESC
        LIMIT 10
        """,
        (f"%{q}%", f"%{q}%"),
    )
    rows = cur.fetchall()
    conn.close()

    if not rows:
        return "No products found."

    formatted = [f"- {name} | {category} | ${price:.2f} | stock: {stock}" for name, category, price, stock in rows]
    return "Matched products:\n" + "\n".join(formatted)


print("Tools defined: read_local_resources, search_online, search_local_database")

### 5. System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are Amr Store Assistant, an AI agent for an e-commerce support desk.

CRITICAL RULES - You MUST follow these exactly:

1) ALWAYS use tools before answering. NEVER guess or make up information.

2) Tool selection guide:
   - read_local_resources: shipping policy, returns policy, refunds, delivery times, international shipping
   - search_local_database: products, stock, inventory, price, availability, keyboards, mouse, monitors, accessories, items for sale
   - search_online: general knowledge, external facts, anything NOT about our store policies or products

3) For product questions (stock, availability, price, items):
   - You MUST call search_local_database FIRST
   - Use simple keywords like "keyboard", "mouse", "monitor", "accessories"
   - Report exactly what the tool returns

4) NEVER say "no products found" or "out of stock" without first calling search_local_database.

5) Be concise and cite your source (tool name or file).

Examples:
- "Do you have keyboards?" -> call search_local_database("keyboard")
- "What's your return policy?" -> call read_local_resources("return")
- "What is the capital of France?" -> call search_online("capital of France")
"""

print("System prompt defined.")

### 6. Create Agent

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    system_prompt=SYSTEM_PROMPT,
    tools=[read_local_resources, search_online, search_local_database],
    checkpointer=InMemorySaver(),
)

print("Agent created successfully.")

### 7. Test Agent - Local Resources Tool

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is your return policy?")]},
    config={"configurable": {"thread_id": "test-1"}},
)
print(response["messages"][-1].content)

### 8. Test Agent - Local Database Tool

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Do you have any keyboards in stock?")]},
    config={"configurable": {"thread_id": "test-2"}},
)
print(response["messages"][-1].content)

### 9. Test Agent - Online Search Tool

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of Japan?")]},
    config={"configurable": {"thread_id": "test-3"}},
)
print(response["messages"][-1].content)